# Gemma 4 E4B for GUI Grounding — OS-Atlas → ScreenSpot-v2

Fine-tunes **Gemma 4 E4B-it** (the vision-capable 4.5B-effective-param variant) on a diverse subset of **OS-Atlas** GUI grounding data and benchmarks it against the **ScreenSpot-v2** test set.

**Why these pieces:**
- Gemma 4's model card lists *screen and UI understanding* and *pointing* as native capabilities — so the base model already has nontrivial grounding ability, which makes the baseline vs fine-tuned comparison meaningful.
- `cua-lite/OS-Atlas` is a preprocessed version of OS-Copilot's OS-Atlas data: images embedded as bytes in parquet, coords pre-normalized to `[0, 1000]` integers, sub-configs per platform, metadata struct for filtering. Loads cleanly via `datasets.load_dataset(streaming=True)` — **no local download to your machine; everything stays in the Kaggle container**.
- `lmms-lab/ScreenSpot-v2` is the test benchmark (1.5k samples, parquet format, images embedded).

**Pipeline:**
1. Load Gemma 4 E4B-it from Kaggle Models (attach `google/gemma-4`, variant `transformers/gemma-4-e4b-it`)
2. Stream OS-Atlas, diverse-sample ~3k examples balanced across platforms / bbox size / position / instruction length
3. Format as Gemma 4 chat with `<image>` + grounding instruction → coord string
4. **Baseline eval** on ScreenSpot-v2 (point-in-bbox accuracy, per-platform, per-element-type)
5. QLoRA fine-tune with Unsloth `FastVisionModel`
6. **Fine-tuned eval** on ScreenSpot-v2; print delta

**Kaggle settings before running:**
- Accelerator: **GPU T4 ×2** (or **L4 ×1** if available) — Internet **ON**
- Models → Add Model → `google/gemma-4` → variant `transformers/gemma-4-e4b-it`
- Optional: add a HuggingFace token as a Kaggle Secret named `HF_TOKEN` (only needed if HF rate-limits you; both datasets are public)


## 1. Install dependencies


In [ ]:
# Unsloth gives us the easiest path to Gemma 4 vision QLoRA on a single Kaggle GPU.
# Pin a recent stack; Gemma 4 needs transformers >= 5.5 and Unsloth's post-April-2026 release.
%%capture
!pip install -q --upgrade pip
!pip install -q 'unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git'
!pip install -q --upgrade 'transformers>=5.5.0' 'trl>=0.21.0' 'peft>=0.14.0' \
    'accelerate>=1.5.0' 'bitsandbytes>=0.45.0' 'datasets>=3.2.0' 'huggingface_hub>=0.27.0' \
    'pillow' 'pyarrow' 'tqdm'


In [ ]:
import os, gc, io, json, math, random, re, time
from collections import defaultdict, Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

# Push HF cache off the small root disk (Kaggle's `/kaggle/working` has ~20GB).
os.environ['HF_HOME'] = '/kaggle/working/.hf'
os.environ['HF_DATASETS_CACHE'] = '/kaggle/working/.hf/datasets'
os.environ['TRANSFORMERS_CACHE'] = '/kaggle/working/.hf/transformers'
Path('/kaggle/working/.hf').mkdir(parents=True, exist_ok=True)

# Optional HF token (only needed if you hit rate limits)
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
    print('HF_TOKEN loaded from Kaggle Secrets')
except Exception:
    print('No HF_TOKEN secret — proceeding anonymously (both datasets are public)')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.device_count(), 'GPU(s)')
for i in range(torch.cuda.device_count()):
    print(' ', torch.cuda.get_device_name(i), '-', round(torch.cuda.get_device_properties(i).total_memory/1e9, 1), 'GB')


## 2. Config


In [ ]:
# ============================== CONFIG ==============================
# Path where Kaggle mounts the attached model. Adjust if Kaggle's path differs
# (the variant slug is sometimes `transformers/gemma-4-E4B-it`).
MODEL_KAGGLE_PATH = '/kaggle/input/gemma-4/transformers/gemma-4-e4b-it/1'
# Fallback: pull from HF if the Kaggle mount isn't found
MODEL_HF_FALLBACK = 'unsloth/gemma-4-E4B-it-bnb-4bit'

# How many examples to keep for fine-tuning (after diverse sampling).
TRAIN_N_TOTAL = 3000
# Pool size to draw from when streaming (larger = more diverse, slower).
STREAM_POOL_PER_PLATFORM = 8000

# Eval subsets (full ScreenSpot-v2 is ~1.5k — fast enough to run all of it).
EVAL_MAX_PER_SPLIT = None  # None = use all; or set e.g. 200 for a quick check

# Training
LORA_R = 16
LORA_ALPHA = 16
LEARNING_RATE = 2e-4
NUM_EPOCHS = 1
PER_DEVICE_BSZ = 1
GRAD_ACCUM = 8
MAX_SEQ_LENGTH = 2048

# Image resizing — keep screenshots large enough that small icons are still resolvable.
# Gemma 4 supports variable resolution; we cap longest side to control memory.
MAX_IMAGE_LONGEST_SIDE = 1024

OUTPUT_DIR = '/kaggle/working/gemma4-osatlas-lora'
SAMPLED_TRAIN_PATH = '/kaggle/working/osatlas_sampled.parquet'
RESULTS_PATH = '/kaggle/working/screenspot_v2_results.json'

print('Config loaded')


## 3. Load Gemma 4 E4B-it

Loaded via Unsloth's `FastVisionModel` (handles 4-bit + LoRA setup for vision models in one shot). On Kaggle, prefer the attached model mount; fall back to HF if the mount path differs.


In [ ]:
from unsloth import FastVisionModel

model_source = MODEL_KAGGLE_PATH if Path(MODEL_KAGGLE_PATH).exists() else MODEL_HF_FALLBACK
print('Loading model from:', model_source)

model, processor = FastVisionModel.from_pretrained(
    model_source,
    load_in_4bit=True,           # QLoRA — required to fit E4B on T4/L4
    use_gradient_checkpointing='unsloth',
    max_seq_length=MAX_SEQ_LENGTH,
)
print('Model loaded.')

# Inference mode by default; we flip to training later before SFT.
FastVisionModel.for_inference(model)


## 4. Grounding prompt + output parser

We use a simple, parseable output format: `<box>x1,y1,x2,y2</box>` in `[0, 1000]` integer coordinates. This matches OS-Atlas's normalized scheme and is robust to parse. For point-in-bbox eval we'll use the bbox center.


In [ ]:
SYSTEM_PROMPT = (
    'You are a GUI grounding model. Given a screenshot and a description of a UI element, '
    'output the bounding box of that element as <box>x1,y1,x2,y2</box> where all coordinates '
    'are integers in [0, 1000] normalized to the image dimensions. Output nothing else.'
)

USER_TEMPLATE = 'Locate this element: "{instruction}"'

BOX_RE = re.compile(r'<box>\s*(\d{1,4})\s*,\s*(\d{1,4})\s*,\s*(\d{1,4})\s*,\s*(\d{1,4})\s*</box>')
# Fallback patterns for when the model doesn't follow the tag format
BOX_RE_LOOSE = re.compile(r'\[?\s*(\d{1,4})\s*,\s*(\d{1,4})\s*,\s*(\d{1,4})\s*,\s*(\d{1,4})\s*\]?')
POINT_RE = re.compile(r'\(\s*(\d{1,4})\s*,\s*(\d{1,4})\s*\)')

def parse_box_1000(text: str):
    """Parse model output → (x1, y1, x2, y2) in [0, 1000], or None."""
    m = BOX_RE.search(text)
    if m is None:
        m = BOX_RE_LOOSE.search(text)
    if m is None:
        # Try a point — convert to a degenerate box
        pm = POINT_RE.search(text)
        if pm is not None:
            x, y = int(pm.group(1)), int(pm.group(2))
            return (x, y, x, y)
        return None
    x1, y1, x2, y2 = (int(m.group(i)) for i in range(1, 5))
    return (x1, y1, x2, y2)

def box_center_1000(box):
    x1, y1, x2, y2 = box
    return ((x1 + x2) / 2, (y1 + y2) / 2)

def point_in_bbox(point_xy, bbox_xyxy):
    """point in [0,1] coords, bbox in [0,1] (x1,y1,x2,y2)."""
    px, py = point_xy
    x1, y1, x2, y2 = bbox_xyxy
    return (x1 <= px <= x2) and (y1 <= py <= y2)

def resize_image(img: Image.Image, longest=MAX_IMAGE_LONGEST_SIDE):
    w, h = img.size
    s = longest / max(w, h)
    if s < 1:
        img = img.resize((int(w * s), int(h * s)), Image.LANCZOS)
    return img.convert('RGB')


## 5. Diverse sampling from OS-Atlas (streaming)

Strategy (no embeddings, fast and reproducible):
1. **Stratify** across `(platform, sub_source)` from cua-lite's metadata struct — guarantees representation of mobile (AMEX), desktop (Windows/Linux/macOS), and web (FineWeb) screenshots.
2. Within each stratum, **stream a pool** of N candidates with shuffle buffer.
3. **Bin** each candidate by (bbox area, bbox quadrant, instruction-length-bucket).
4. **Round-robin** across bins so we get small icons + large regions, all quadrants, short labels + long descriptions.

We avoid materializing the full 4.7M rows — streaming pulls only what we sample.


In [ ]:
from datasets import load_dataset

# cua-lite subsets we want — each is a (platform, task_type) cohort.
PLATFORM_CONFIGS = [
    'desktop-grounding-bbox',  # 1.14M rows — Windows / Linux / macOS
    'mobile-grounding-bbox',   # 1.58M rows — AMEX (Android)
    'web-grounding-bbox',      # web — FineWeb
]

def extract_bbox_and_instruction_from_messages(messages):
    """cua-lite stores the conversation in `messages` field.
    User turn has the instruction; assistant turn has the bbox string.
    We pull both. bbox is integers in [0, 1000]."""
    user_text, assistant_text = None, None
    for m in messages:
        # `content` may be a list of dicts (multimodal) or a string
        if m.get('role') == 'user':
            c = m.get('content')
            if isinstance(c, list):
                user_text = ' '.join(p.get('text', '') for p in c if p.get('type') == 'text')
            else:
                user_text = c
        elif m.get('role') == 'assistant':
            c = m.get('content')
            assistant_text = c if isinstance(c, str) else ' '.join(p.get('text', '') for p in c if isinstance(p, dict))
    if user_text is None or assistant_text is None:
        return None, None
    box = parse_box_1000(assistant_text)
    return user_text, box

def bin_features(box_1000, instr_text, img_w_or_none=None):
    """Return a tuple of bins for diversity routing."""
    x1, y1, x2, y2 = box_1000
    # Area bin (out of 1000*1000 = 1e6)
    area = max(0, (x2 - x1)) * max(0, (y2 - y1))
    if area < 2000:           area_bin = 'tiny'      # small icons, ~<4% area
    elif area < 20000:        area_bin = 'small'
    elif area < 100000:       area_bin = 'medium'
    else:                     area_bin = 'large'
    # Position quadrant (use center)
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    quad = ('L' if cx < 500 else 'R') + ('T' if cy < 500 else 'B')
    # Instruction length bin
    nw = len((instr_text or '').split())
    if nw <= 3:               len_bin = 'short'
    elif nw <= 8:             len_bin = 'mid'
    else:                     len_bin = 'long'
    return (area_bin, quad, len_bin)


In [ ]:
# Sample diversely per platform-cohort, then merge.
samples = []
per_platform_target = TRAIN_N_TOTAL // len(PLATFORM_CONFIGS)

for cfg in PLATFORM_CONFIGS:
    print(f'\n=== Streaming {cfg} (target={per_platform_target}) ===')
    ds = load_dataset('cua-lite/OS-Atlas', cfg, split='train', streaming=True)
    ds = ds.shuffle(buffer_size=4096, seed=SEED)

    bin_counts = Counter()
    bin_buckets = defaultdict(list)
    pool_seen = 0

    for ex in ds:
        pool_seen += 1
        if pool_seen > STREAM_POOL_PER_PLATFORM:
            break
        # cua-lite schema: ex has 'messages' (with bbox in text) and possibly 'images'.
        instr, box = extract_bbox_and_instruction_from_messages(ex.get('messages', []))
        if box is None or instr is None:
            continue
        # Get the image — cua-lite embeds bytes in the 'images' field
        imgs = ex.get('images') or ex.get('image')
        if imgs is None:
            continue
        if isinstance(imgs, list):
            if len(imgs) == 0: continue
            img = imgs[0]
        else:
            img = imgs
        if isinstance(img, dict) and 'bytes' in img:
            try:
                img = Image.open(io.BytesIO(img['bytes']))
            except Exception:
                continue
        # Standardize: degenerate boxes (zero area) are noise — skip
        x1, y1, x2, y2 = box
        if x2 <= x1 or y2 <= y1:
            continue
        bin_key = bin_features(box, instr)
        meta = ex.get('metadata') or {}
        sub_source = (meta.get('others') or {}).get('sub_source') or meta.get('sub_source') or 'unk'
        bin_buckets[bin_key].append({
            'platform_cohort': cfg,
            'sub_source': sub_source,
            'instruction': instr,
            'bbox_1000': list(box),
            'image': img,  # PIL — we'll re-encode at write time
        })
        bin_counts[bin_key] += 1

    # Round-robin draw from bins until we hit the per-platform target
    print(f'  pool_seen={pool_seen}, populated_bins={len(bin_buckets)}')
    bins = list(bin_buckets.keys())
    random.shuffle(bins)
    drawn = 0
    while drawn < per_platform_target and any(bin_buckets[b] for b in bins):
        for b in bins:
            if drawn >= per_platform_target: break
            if bin_buckets[b]:
                samples.append(bin_buckets[b].pop())
                drawn += 1
    print(f'  drawn={drawn}')

print(f'\nTotal sampled: {len(samples)}')
print('Cohort balance:', Counter(s['platform_cohort'] for s in samples))
print('Sub-source balance:', Counter(s['sub_source'] for s in samples))


In [ ]:
# Quick visualization of one sample to sanity-check
import matplotlib.pyplot as plt
import matplotlib.patches as patches

s = samples[0]
img = s['image']
w, h = img.size
x1, y1, x2, y2 = s['bbox_1000']
x1p, y1p, x2p, y2p = x1/1000*w, y1/1000*h, x2/1000*w, y2/1000*h
fig, ax = plt.subplots(figsize=(8, 5))
ax.imshow(img)
ax.add_patch(patches.Rectangle((x1p, y1p), x2p-x1p, y2p-y1p, fill=False, edgecolor='red', linewidth=2))
ax.set_title(f"{s['platform_cohort']} | {s['sub_source']}\n{s['instruction'][:80]}")
ax.axis('off')
plt.show()


## 6. Persist the sampled subset

Save to a parquet in `/kaggle/working` so you can attach it as a dataset to other notebooks and so reruns of training don't re-stream.


In [ ]:
# Re-encode PIL images to bytes, store as parquet
def to_bytes(img: Image.Image, fmt='PNG'):
    buf = io.BytesIO()
    resize_image(img).save(buf, format=fmt)
    return buf.getvalue()

rows = []
for s in tqdm(samples, desc='Encoding'):
    rows.append({
        'platform_cohort': s['platform_cohort'],
        'sub_source': s['sub_source'],
        'instruction': s['instruction'],
        'bbox_1000': s['bbox_1000'],
        'image_bytes': to_bytes(s['image']),
    })
df = pd.DataFrame(rows)
df.to_parquet(SAMPLED_TRAIN_PATH, index=False)
print('Saved', SAMPLED_TRAIN_PATH, '|', len(df), 'rows |',
      round(Path(SAMPLED_TRAIN_PATH).stat().st_size/1e6, 1), 'MB')
del samples; gc.collect()


## 7. Load ScreenSpot-v2 test set

Use `lmms-lab/ScreenSpot-v2` which is parquet-formatted with images embedded (cleanest loader). It has three splits — desktop, mobile, web — each with `text` and `icon` element types.


In [ ]:
screenspot = load_dataset('lmms-lab/ScreenSpot-v2')
print(screenspot)
# Peek schema
first_split = list(screenspot.keys())[0]
print('\nFirst-split features:', screenspot[first_split].features)
print('First row keys:', list(screenspot[first_split][0].keys()))
print('First row (sans image):', {k: v for k, v in screenspot[first_split][0].items() if k != 'image'})


## 8. ScreenSpot-v2 evaluation function

Standard metric: model predicts a bbox; we take its center; correct iff the center falls inside the ground-truth bbox. Reported per platform and per `(platform, element_type)`.


In [ ]:
from transformers import TextStreamer

def build_chat(image, instruction):
    """Build messages for Gemma 4 chat template. Image MUST come before text."""
    return [
        {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]},
        {'role': 'user', 'content': [
            {'type': 'image', 'image': image},
            {'type': 'text', 'text': USER_TEMPLATE.format(instruction=instruction)},
        ]},
    ]

@torch.inference_mode()
def predict_box(image, instruction, max_new_tokens=64):
    image = resize_image(image)
    messages = build_chat(image, instruction)
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors='pt',
    ).to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=1.0,
        use_cache=True,
    )
    gen = out[0][inputs['input_ids'].shape[1]:]
    text = processor.decode(gen, skip_special_tokens=True)
    return text, parse_box_1000(text)

def normalize_screenspot_bbox(bbox, w, h):
    """ScreenSpot bbox formats vary; normalize to (x1, y1, x2, y2) in [0, 1]."""
    x1, y1, x2_or_w, y2_or_h = bbox
    # Heuristic: if max value > 1.5 it's pixel coords. Then guess (x1, y1, x2, y2) vs (x, y, w, h).
    if max(bbox) <= 1.01:  # already [0, 1]
        # Could still be xywh — assume xyxy unless inverted
        if x2_or_w < x1 or y2_or_h < y1:
            return (x1, y1, x1 + x2_or_w, y1 + y2_or_h)
        return (x1, y1, x2_or_w, y2_or_h)
    # Pixel coords
    # Decide xyxy vs xywh: if x2 <= x1 it's xywh
    if x2_or_w <= x1 or y2_or_h <= y1:
        px1, py1, px2, py2 = x1, y1, x1 + x2_or_w, y1 + y2_or_h
    else:
        px1, py1, px2, py2 = x1, y1, x2_or_w, y2_or_h
    return (px1/w, py1/h, px2/w, py2/h)

def evaluate_screenspot(ds_dict, name, max_per_split=None):
    results = []
    for split_name, split in ds_dict.items():
        rng = list(range(len(split)))
        if max_per_split is not None:
            rng = rng[:max_per_split]
        for i in tqdm(rng, desc=f'{name}/{split_name}'):
            ex = split[i]
            img = ex['image']
            if not isinstance(img, Image.Image):
                img = Image.open(io.BytesIO(img['bytes']))
            img = img.convert('RGB')
            w, h = img.size
            instruction = ex.get('instruction') or ex.get('text') or ex.get('query')
            data_type = ex.get('data_type') or ex.get('element_type') or 'unk'
            data_source = ex.get('data_source') or ex.get('platform') or split_name
            gt_bbox_raw = ex.get('bbox') or ex.get('bounding_box')
            gt_box_01 = normalize_screenspot_bbox(gt_bbox_raw, w, h)
            text, pred = predict_box(img, instruction)
            if pred is None:
                ok = False
                pc = (None, None)
            else:
                cx, cy = box_center_1000(pred)
                pc = (cx/1000, cy/1000)
                ok = point_in_bbox(pc, gt_box_01)
            results.append({
                'split': split_name, 'data_type': data_type, 'data_source': data_source,
                'instruction': instruction, 'gt_bbox_01': gt_box_01,
                'pred_raw': text, 'pred_box_1000': pred,
                'pred_center_01': pc, 'correct': ok,
            })
    return pd.DataFrame(results)

def report(df, tag):
    print(f'\n========== {tag} ==========')
    print(f'Overall accuracy: {df.correct.mean():.4f}  (n={len(df)})')
    print('\nBy split:')
    print(df.groupby('split').correct.agg(['mean', 'count']).round(4))
    print('\nBy (split, data_type):')
    print(df.groupby(['split', 'data_type']).correct.agg(['mean', 'count']).round(4))


## 9. Baseline evaluation


In [ ]:
FastVisionModel.for_inference(model)
baseline_df = evaluate_screenspot(screenspot, 'baseline', max_per_split=EVAL_MAX_PER_SPLIT)
report(baseline_df, 'BASELINE (Gemma 4 E4B-it, no fine-tune)')
baseline_df.to_parquet('/kaggle/working/screenspot_v2_baseline.parquet', index=False)


## 10. Build the training dataset

Convert each sampled row → Gemma 4 multimodal chat with the bbox as the assistant target. Loss is masked to assistant tokens only (Unsloth handles this with `train_on_responses_only`).


In [ ]:
from datasets import Dataset

def row_to_messages(row):
    img = Image.open(io.BytesIO(row['image_bytes'])).convert('RGB')
    img = resize_image(img)
    x1, y1, x2, y2 = row['bbox_1000']
    target = f'<box>{x1},{y1},{x2},{y2}</box>'
    return {
        'messages': [
            {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]},
            {'role': 'user', 'content': [
                {'type': 'image', 'image': img},
                {'type': 'text', 'text': USER_TEMPLATE.format(instruction=row['instruction'])},
            ]},
            {'role': 'assistant', 'content': [{'type': 'text', 'text': target}]},
        ],
    }

df = pd.read_parquet(SAMPLED_TRAIN_PATH)
print('Building chat-format dataset from', len(df), 'rows')
train_records = [row_to_messages(r) for _, r in tqdm(df.iterrows(), total=len(df))]
train_ds = Dataset.from_list(train_records)
print(train_ds)


## 11. Fine-tune with Unsloth + TRL


In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.trainer import UnslothVisionDataCollator

FastVisionModel.for_training(model)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,      # keep SigLIP frozen — grounding lives in the LM
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias='none',
    random_state=SEED,
    use_rslora=False,
)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=PER_DEVICE_BSZ,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    optim='adamw_8bit',
    weight_decay=0.0,
    logging_steps=10,
    save_strategy='epoch',
    save_total_limit=1,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    report_to='none',
    seed=SEED,
    # Vision-specific knobs
    remove_unused_columns=False,
    dataset_text_field='',
    dataset_kwargs={'skip_prepare_dataset': True},
    max_seq_length=MAX_SEQ_LENGTH,
)

trainer = SFTTrainer(
    model=model,
    processing_class=processor.tokenizer,
    data_collator=UnslothVisionDataCollator(model, processor),
    train_dataset=train_ds,
    args=training_args,
)

trainer_stats = trainer.train()
print(trainer_stats)


In [ ]:
# Save the LoRA adapter
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print('Adapter saved to', OUTPUT_DIR)


## 12. Re-evaluate on ScreenSpot-v2


In [ ]:
FastVisionModel.for_inference(model)
finetuned_df = evaluate_screenspot(screenspot, 'finetuned', max_per_split=EVAL_MAX_PER_SPLIT)
report(finetuned_df, 'FINE-TUNED (Gemma 4 E4B-it + OS-Atlas LoRA)')
finetuned_df.to_parquet('/kaggle/working/screenspot_v2_finetuned.parquet', index=False)


## 13. Baseline vs Fine-tuned comparison


In [ ]:
def summary(df):
    overall = df.correct.mean()
    by_split = df.groupby('split').correct.mean()
    by_split_type = df.groupby(['split', 'data_type']).correct.mean()
    return overall, by_split, by_split_type

b_overall, b_split, b_split_type = summary(baseline_df)
f_overall, f_split, f_split_type = summary(finetuned_df)

print(f'Overall: baseline={b_overall:.4f} → finetuned={f_overall:.4f}  (Δ={(f_overall-b_overall)*100:+.2f} pp)')
print('\nBy split:')
cmp = pd.DataFrame({'baseline': b_split, 'finetuned': f_split})
cmp['delta_pp'] = (cmp.finetuned - cmp.baseline) * 100
print(cmp.round(4))
print('\nBy (split, data_type):')
cmp2 = pd.DataFrame({'baseline': b_split_type, 'finetuned': f_split_type})
cmp2['delta_pp'] = (cmp2.finetuned - cmp2.baseline) * 100
print(cmp2.round(4))

with open(RESULTS_PATH, 'w') as f:
    json.dump({
        'baseline_overall': float(b_overall),
        'finetuned_overall': float(f_overall),
        'baseline_by_split': b_split.to_dict(),
        'finetuned_by_split': f_split.to_dict(),
    }, f, indent=2)
print('Saved', RESULTS_PATH)


## 14. Notes / next experiments

- **If baseline ≈ fine-tuned**: try (a) larger `TRAIN_N_TOTAL` (5–10k), (b) unfreezing the vision tower for the last few epochs, (c) increasing `MAX_IMAGE_LONGEST_SIDE` to 1280 for small-icon discrimination on web/desktop.
- **If fine-tuned regresses on one split**: the diverse sampler may have over-represented a different cohort. Check `Counter(s['platform_cohort'] for s in samples)` and the per-`sub_source` breakdown — manually re-balance the `per_platform_target`s.
- **Output format**: `<box>x1,y1,x2,y2</box>` is parser-friendly but center-point output (`<click>x,y</click>`) often wins on ScreenSpot-style benchmarks. OS-Atlas-Base-7B uses both — easy to swap by changing `SYSTEM_PROMPT` and the target string in `row_to_messages`.
- **Failure analysis**: `baseline_df` and `finetuned_df` are saved as parquet — load them, filter `correct==False`, and look at the screenshots to find systematic failure modes (e.g. dense menus, low-contrast icons).
- **Comparison against OS-Atlas-Base-4B/7B** (Qwen2-VL backbone): drop in `OS-Copilot/OS-Atlas-Base-4B` via `AutoModelForCausalLM` and run the same `evaluate_screenspot` for a 3-way comparison.
